In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-kaizen

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 6.3: Parallel Execution + LLM-Based Routing
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Run independent specialists concurrently with asyncio.gather
#   - Prove parallel latency ≈ max(stages), not sum
#   - Use Kaizen Pipeline.router() for LLM-based dispatch
#
# PREREQUISITES: 02_sequential_pipeline.py
# ESTIMATED TIME: ~30 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import time

from kaizen_agents import Pipeline

from shared.mlfp06.ex_6 import (
    OUTPUT_DIR,
    build_specialists,
    load_squad_corpus,
)



## TASK 1 — Load corpus + specialists


In [ ]:
# TODO: Load corpus and build the three specialists
passages = ____
factual_agent, semantic_agent, structural_agent = ____



### Checkpoint 1


In [ ]:
assert passages.height > 0
assert factual_agent and semantic_agent and structural_agent
print("✓ Checkpoint 1 passed\n")



## TASK 2 — Parallel orchestrator with asyncio.gather


Launch all specialists simultaneously with asyncio.gather.


Same work, but one agent at a time — for latency comparison.


In [ ]:
async def parallel_analysis(doc: str, question: str) -> dict:
    t0 = time.perf_counter()

    # TODO: Build three coroutine tasks (do NOT await them yet).
    # Hint: BaseAgent.run_async is a coroutine — calling it without `await`
    # returns the coroutine object you can pass to asyncio.gather.
    #   factual_task = factual_agent.run_async(document=doc, question=question)
    factual_task = ____
    semantic_task = ____
    structural_task = ____

    # TODO: Await all three concurrently with asyncio.gather(...)
    factual_r, semantic_r, structural_r = ____

    # run_async returns a dict — read OutputField values with dict indexing.
    return {
        "factual_claims": factual_r["factual_claims"],
        "themes": semantic_r["main_themes"],
        "entities": structural_r["key_entities"],
        "latency_s": time.perf_counter() - t0,
    }


async def sequential_baseline(doc: str, question: str) -> float:
    t0 = time.perf_counter()
    # TODO: await the three specialists one at a time, using run_async.
    # Hint: await factual_agent.run_async(document=doc, question=question)
    ____
    ____
    ____
    return time.perf_counter() - t0



## TASK 3 — Measure parallel vs sequential latency


In [ ]:
doc = passages["text"][0]
question = passages["question"][0]


async def run_comparison():
    par = await parallel_analysis(doc, question)
    seq_latency = await sequential_baseline(doc, question)
    return par, seq_latency


par_result, seq_latency  = await run_comparison()



### Checkpoint 2


In [ ]:
assert par_result["factual_claims"], "Parallel run should produce claims"
assert par_result["latency_s"] <= seq_latency, "Parallel should not be slower"
print("✓ Checkpoint 2 passed — parallel execution verified\n")



## TASK 4 — Configure Pipeline.router() for LLM-based dispatch


In [ ]:
# TODO: Build an LLM router over the three specialists.
# Hint: Pipeline.router(agents=[factual_agent, semantic_agent, structural_agent])
router = ____



### Checkpoint 3


In [ ]:
assert router is not None, "Task 4: router should be created"
print("✓ Checkpoint 3 passed — router configured\n")



## TASK 5 — Intent-to-specialist mapping


In [ ]:
test_queries = [
    ("What specific dates and numbers are mentioned?", "factual"),
    ("What is the underlying theme of the argument?", "semantic"),
    ("How is the passage organised, and what entities appear?", "structural"),
]

for query, expected in test_queries:
    print(f"  Query: {query}")
    print(f"    Expected specialist: {expected}")



## APPLY — Singapore scenario: Smart Nation helpdesk triage


In [ ]:
# A Singapore government agency runs a citizen helpdesk at 4,000
# tickets/day. Keyword routing mis-routes 18%; LLM routing drops
# that to 3%. At an average rework cost of 8 minutes per mis-route,
# this reclaims ~80 hours/day — ~S$700K/year in agent time.



## REFLECTION


[x] asyncio.gather for concurrent specialist execution
  [x] Parallel latency ≈ max(stages) vs sequential = sum
  [x] Pipeline.router() — LLM-based dispatch by capability card

  Next: 04_mcp_server.py — exposing tools via MCP so any compatible
  agent can discover and call them.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

